# ML Week 11: Neural Networks

With the basics of vectors and matrices we learned last week, we already have the skills to build a neural network!

We will start with a simple one:

* Input Layer = 4 neurons
* Hidden Layer 1 = 4 neurons
* Hidden Layer 2 = 4 neurons
* Output Layer = 4 neurons

<div>
<img src="./NN1.png" width="1000"/>
</div>
Source: https://alexlenail.me/NN-SVG/

Each layer will be a vector (where each number is a node).

The weights between the nodes will be a matrix:

$$\begin{bmatrix} w_{11} & w_{12} & w_{13} & w_{14} \\ w_{21} & w_{22} & w_{23} & w_{24} \\ w_{31} & w_{32} & w_{33} & w_{34} \\ w_{41} & w_{42} & w_{43} & w_{44}\end{bmatrix}$$

In this matrix, each entry represents the "weight" of a connection. For example $w_{31}$ is the strenght of the connection from the 3rd node to the 1st node.

### Let's try connecting the first layer!

In [3]:
import numpy as np
import scipy as sp

In [51]:
# Starting with some generic input (the input layer)

input = np.array([[1,2,3,4]])
print(input)

[[1 2 3 4]]


In [52]:
# Now let's make some random weights. In modern neural networks weights are usually initialized to be small and between -1 and +1
weights = np.random.uniform(-1,1,(4,4))
print(weights)

# Note: In fact, there are slightly more clever ways of intitializing weights (ie "He Initialization"), but we won't worry about that for simplicity

[[-0.54814982  0.97732887  0.49635368 -0.25261505]
 [ 0.44914135 -0.29880846  0.51298164 -0.25868722]
 [-0.8673944  -0.36393358  0.39563952  0.84765598]
 [ 0.76013463 -0.63892628  0.00896807  0.28395363]]


In [53]:
# Let's propogate the signal from the input to the first hidden layer
hidden_layer1 = input @ weights
print(hidden_layer1)

[[ 0.78848822 -3.26779392  2.74510778  2.90879299]]


### Activation Functions

It turns out that networks built ONLY on repeated multiplication and addition are not very good at capturing more commplex behaviour.

Very early on it was realized that adding one extra step at each layer truly unlocks the power of neural networks.

This is adding what are called "Activation Functions".

There are many different versions, but now the industry standard is called "ReLU" which stands for "Rectified Linear Function".

This is actually super simple! For each hidden layer, before continuing on to the next one, set all negative neurons to zero.

In code, it is simply the super easy addition of one more line:

In [54]:
hidden_layer1 = input @ weights
hidden_layer1 = np.maximum(hidden_layer1,0) # ReLu Activation Function
print(hidden_layer1)

[[0.78848822 0.         2.74510778 2.90879299]]


### Computing Multiple Layers

In [55]:
# We can also put multiple layers into a single matrix.
# This matrix has 3 layers of 4x4 weight values:

weights = np.random.uniform(-1, 1, (3,4,4))

In [56]:
weights

array([[[ 0.1935953 , -0.40454813,  0.73973511, -0.83879824],
        [ 0.52926388, -0.7532368 , -0.54937855,  0.21724649],
        [ 0.47581968, -0.23573587,  0.20476368, -0.7892931 ],
        [-0.69190075,  0.00813594,  0.72851707, -0.88025214]],

       [[-0.72254438,  0.57635685,  0.8928373 , -0.9721684 ],
        [-0.28205394,  0.63639615, -0.22441541, -0.86355037],
        [-0.22241692,  0.10017184, -0.87480403,  0.76123944],
        [-0.85263546, -0.88763775,  0.09843449, -0.76328151]],

       [[ 0.28925329,  0.22036797,  0.95068607, -0.62029346],
        [-0.10518619, -0.93045568, -0.08810865,  0.43813526],
        [ 0.5671918 , -0.34039164,  0.17566349,  0.60433929],
        [-0.69474108, -0.66302425,  0.72411141,  0.0506921 ]]])

In [57]:
# We do exactly the same thing, but this time we need to specify which section (or "slice") of the 3D matrix we want to use

hidden_layer1 = input @ weights[0]  # We do that here with the index on the end
hidden_layer1 = np.maximum(hidden_layer1,0)
hidden_layer1

array([[0.        , 0.        , 3.16933732, 0.        ]])

In [58]:
# We can also put this into a loop where each iteration chooses the next batch of weights:

for i in range(weights.shape[0]):
    hidden_layer = input @ weights[i]
    hidden_layer = np.maximum(hidden_layer,0)
    input = hidden_layer
    print(hidden_layer)

[[0.         0.         3.16933732 0.        ]]
[[0.         0.31747834 0.         2.41262457]]
[[0.         0.         1.7190364  0.26139946]]


### But how do we choose the selected class from the final layer?

The final layer is just a bunch of numbers. How do we decide which class has been selected?

For this, we first use something called "softmax". This changes the numbers into probabilities of the input belonging to each of the classes.

So if the final layer is $[0.1, 0.6, 0.1, 0.2]$ then this means there is a 10% chance of the input belonging to class 1, a 60% input of the input belonging to class 2, a 10% chance of the input belonging to class 3, and a 20% chance of the input belonging to class 4.

Lastly, to choose the most probable class we use the "argmax" (maximum argumnet) method:

In [59]:
for i in range(weights.shape[0]):
    hidden_layer = input @ weights[i]  # Multiply (dot product) with the weights
    hidden_layer = np.maximum(hidden_layer,0)  # ReLU
    input = hidden_layer  # Set the current layer as input to the next one and repeat
    print(hidden_layer)

output = sp.special.softmax(hidden_layer)
np.argmax(output)

[[0.63708888 0.         0.54243018 0.        ]]
[[0.         0.42152677 0.0942966  0.        ]]
[[0.00914546 0.         0.         0.24167288]]


np.int64(3)

In [ ]:
# We can also put this into a function:

def forward_pass(input, weights):
    for i in range(weights.shape[0]):
        input = input @ weights[i,:,:]
        input = np.maximum(input,0)  # This time we just overwrite input repeatedly as we don't really need an extra "hidden_layer" variable. More efficient.
    
    output = sp.special.softmax(input)
    output_class = np.argmax(output)
    
    return output_class

In [ ]:
# Running the function with many different inputs we see different inputs are classified as different classes
# Note that it's normal to have inbalanced classes - we randomly initialized the weights.

for i in range(50):
    print(forward_pass(np.random.uniform(-1,1,(1,4)), weights))  # Run the forward pass with some random input and the weights from before.

3
0
2
2
3
2
2
0
3
3
2
2
3
3
3
3
3
3
2
3
2
3
3
3
3
3
2
3
0
3
3
3
0
0
2
3
0
3
3
2
3
3
3
3
0
3
3
2
2
2


### This simplest form of Artificial Neural Network (ANN) is called a Multi-Layer Perceptron (MLP).

### Okay...but these weights are random. How do we actually train them?

By giving a network a bunch of input (X) for which you know the expected classes (y) you can repeatedly adjust the weights based on how "good or bad" the guessed output was. The adjustments are propagated from the last layer to the first layer "backwards", hence this training is called "back propagation" (or "backprop").

In fact, the math behind backprop is not that bad, but understanding it perfectly is probably too much for this course.

# Exercises

### Exercise 1

Multi-Layer Perceptrons are easy to set up with scikit-learn, even if they do have a lot of adjustable things (hyperparametres).

As with previous ML methods, MLPs can be used for either Classification (MLPClassifier) or Regression (MLPRegressor).

Below we read in the Wanyika data we used last semester. Let's try to build a regressor to predict site date from the available tabular features.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# load site data
df = pd.read_csv('./Wanyika_cleaned.csv')

# set the y (the label we want to predict)
y = df['Mean Chronology  (Calibrated BP - 1950) ']

# set the X (the features we want the algorithm to learn from)
X = df.drop(columns=['Mean Chronology  (Calibrated BP - 1950) '])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [ ]:
# MLP = Multi-Layer Perceptron

from sklearn.neural_network import MLPRegressor
mlp_reg = MLPRegressor(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 5, 2), max_iter=10000)  # lbfgs = Limited-memory Broyden–Fletcher–Goldfarb–Shanno

In [ ]:
mlp_reg.fit(X_train,y_train)

,loss,'squared_error'
,hidden_layer_sizes,"(5, ...)"
,activation,'relu'
,solver,'lbfgs'
,alpha,1e-05
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,10000
,shuffle,True


In [ ]:
pred = mlp_reg.predict(X_test)
mean_absolute_error(y_test, pred)

954.270463883602

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

# set up the classifier 
mlp_reg = MLPRegressor(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 5, 2), max_iter=10000)

# set the number of folds
k_folds = KFold(n_splits = 4, shuffle=True)

# calculate the scores
scores = cross_val_score(mlp_reg, X, y, cv = k_folds, scoring="neg_mean_absolute_error")

# print the scores
print("Cross Validation Scores: ", scores)
print("Average CV Score: ", scores.mean())
print("Standard deviation: ", scores.std())

Cross Validation Scores:  [-947.93053111 -758.96509042 -802.18652156 -760.32105691]
Average CV Score:  -817.3508000019021
Standard deviation:  77.3665047004174


In [109]:
from sklearn.model_selection import KFold, cross_val_score

# set up the classifier 
mlp_reg = MLPRegressor(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(20, 20, 2), max_iter=10000)

# set the number of folds
k_folds = KFold(n_splits = 4, shuffle=True)

# calculate the scores
scores = cross_val_score(mlp_reg, X, y, cv = k_folds, scoring="neg_mean_absolute_error")

# print the scores
print("Cross Validation Scores: ", scores)
print("Average CV Score: ", scores.mean())
print("Standard deviation: ", scores.std())

Cross Validation Scores:  [-798.42912323 -919.8959363  -977.96510858 -814.48521757]
Average CV Score:  -877.6938464223879
Standard deviation:  74.35315983536131


### Exercise 2

The test network we built had input, hidden, and output layers all with 4 neurons.

This makes them very clean to code, but not very realistic. In general we have larger hidden layers, and smaller output layers.

Let's build a more realistic network that takes in 5 input features, has 3 hidden layers each with 10 neurons, and finally classifies between 3 classes (3 output neurons).

The network will look something like this:

<div>
<img src="./NN2.png" width="1000"/>
</div>
Source: https://alexlenail.me/NN-SVG/

In [33]:
def forward_pass(input, input_weights, hidden_weights, output_weights):

    input = input @ input_weights  # New: A first layer to get the input to the right shape for the hidden layers
    
    for i in range(hidden_weights.shape[0]):
        input = input @ hidden_weights[i,:,:]
        input = np.maximum(input,0)
    
    output = input @ output_weights  # New: A final layer to get the output of the hidden layer to the right shape for the output layer
    output = sp.special.softmax(output)
    output_class = np.argmax(output)
    
    return output_class

In [ ]:
input_weights = np.random.uniform(-1, 1, (5,10))
hidden_weights = np.random.uniform(-1, 1, (2,10,10))
output_weights = np.random.uniform(-1, 1, (10,3))

for i in range(50):
    print(forward_pass(np.random.uniform(-1,1,(1,5)), input_weights, hidden_weights, output_weights))  # Run the forward pass with some random input and the weights from before.

2
0
2
2
0
2
2
1
1
2
1
1
2
2
2
2
2
2
2
1
1
1
1
1
2
1
1
2
2
1
2
2
1
0
2
0
2
2
2
1
2
2
1
2
2
2
2
1
2
2
